<a href="https://colab.research.google.com/github/sophiegsmith/sentiment-product-recommendation/blob/main/02_product_reccomendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Necessary Libraries

In [ ]:
%pip install -q gensim

In [ ]:
import pandas as pd
import numpy as np
import re
from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import json
from gensim.models import Word2Vec
import os
import urllib.request
import zipfile

# Load Cleaned Dataset

In [ ]:
uploaded = files.upload()

Saving product_analysis.csv to product_analysis.csv
Saving reviews_clean.csv to reviews_clean.csv


In [ ]:
reviews_clean = pd.read_csv('reviews_clean.csv')
product_analysis = pd.read_csv('product_analysis.csv')

#check if the csvs are loading
print("Cleaned Reviews: ", reviews_clean.shape)
print("Proudct in Analysis: ", product_analysis.shape)

Cleaned Reviews:  (10188, 8)
Proudct in Analysis:  (239, 13)


In [ ]:
def clean_product_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

product_analysis["product_text"] = (
    product_analysis["title"].fillna("") + " " +
    product_analysis["store"].fillna("")
).apply(clean_product_text)

print(product_analysis[["title", "product_text"]].head())

                                               title  \
0  Retinol Face Moisturizer Cream Natural Facial ...   
1  OZNaturals Facial Mask - This Moisturizing Fac...   
2  Argan Magic 100% Pure Argan Oil – Moisturizing...   
3  Pure Hyaluronic Acid Serum with Vitamin C for ...   
4  Nadira Organics Virgin Argan Oil for Skin, Fac...   

                                        product_text  
0  retinol face moisturizer cream natural facial ...  
1  oznaturals facial mask this moisturizing face ...  
2  argan magic 100 pure argan oil moisturizing oi...  
3  pure hyaluronic acid serum with vitamin c for ...  
4  nadira organics virgin argan oil for skin face...  


# Convert Product Text Into TF-IDF Vectors

In [ ]:
#create TF-IDF Vectors
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(product_analysis["product_text"])
print("TF-IDF matrix shape: ", tfidf_matrix.shape)

TF-IDF matrix shape:  (239, 1767)


# Calculate Product Similarity

In [ ]:
#calculate pairwise cosine similarity between products
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity matrix shape: ", similarity_matrix.shape)

Similarity matrix shape:  (239, 239)


# Create Function to find similar products

In [ ]:
def get_similar_products(product_index, n = 5):
  #get similarity scores for the selected product
  similarity_scores = similarity_matrix[product_index]
  #sort products by similarity, highest first
  similar_indices = similarity_scores.argsort()[::-1]
  #remove the product itself
  similar_indices = similar_indices[similar_indices != product_index]
  #select the top n similar products
  top_indices = similar_indices[:n]
  #return useful product information
  results = product_analysis.iloc[top_indices][
      [
            "item_id",
            "title",
            "store",
            "review_count",
            "avg_review_rating",
            "positive_pct",
            "weighted_rating"
      ]
  ].copy()
  # Add similarity score
  results["similarity_score"] = similarity_scores[top_indices]
  return results

# Test Product Similarity

In [ ]:
# Find the product index for Natwag Makeup Remover Pads
target_index = product_analysis[
    product_analysis["title"].str.contains(
        "Natwag Makeup Remover Pads",
        case=False,
        na=False
    )
].index[0]

print("Target product:")
print(product_analysis.loc[target_index, "title"])

Target product:
Natwag Makeup Remover Pads,18 Pack 3.74 Inch Reusable Microfiber Cleansing Pad with Laundry Bag Suitable for All Skin Types,3 Color Facial Cleansing Pad Set.


In [ ]:
similar_products = get_similar_products(target_index, n=5)

print(similar_products.to_string(index=False))

   item_id                                                                                                                                                                                         title                store  review_count  avg_review_rating  positive_pct  weighted_rating  similarity_score
B088LWRYJC NATWAG Reusable Update Makeup Remover Cloths for Sensitive Skin - Microfiber Face Cloth Makeup Remover | Washable Round Makeup Remover Pads | Natural Facial Towels (2 Pads,2 Cloth) (Purple)               Natwag            17           4.823529     94.117647         4.625980          0.416596
B0B6653VGM                               A24 Cleansing Oil - Makeup Remover, Deep Cleansing, Non-Stripping, 99.75% Natural Ingredients, Vegan Formula, Ideal for All Skin Types… (Cleansing Oil to Foam)                  A24            10           4.600000     80.000000         4.445073          0.221755
B0851C4YPC                  [the SAEM] Derma Plan Cleansing Bubble Pad 15pcs - One Step 

# Adding Recommendatin Logic

## Components of the Recommendation

**1** Similarity Score --> is it comparable?

**2** Weighted Rating --> does it have strong customer feedback

**3** Positive Sentiment --> are customers generally satisfied

In [ ]:
def get_recommendations(product_index, n=5, min_similarity=0.10):
    # Get similarity scores for the target product
    similarity_scores = similarity_matrix[product_index]

    # Rank products by similarity
    similar_indices = similarity_scores.argsort()[::-1]

    # Remove the target product itself
    similar_indices = similar_indices[similar_indices != product_index]

    # Create candidate recommendations
    recommendations = product_analysis.iloc[similar_indices].copy()
    recommendations["similarity_score"] = similarity_scores[similar_indices]

    # Keep only reasonably similar products
    recommendations = recommendations[
        recommendations["similarity_score"] >= min_similarity
    ].copy()

    # Normalize rating and sentiment
    recommendations["rating_score"] = (
        recommendations["weighted_rating"] / 5
    )

    recommendations["sentiment_score"] = (
        recommendations["positive_pct"] / 100
    )

    # Combined recommendation score
    recommendations["recommendation_score"] = (
        0.50 * recommendations["similarity_score"] +
        0.30 * recommendations["rating_score"] +
        0.20 * recommendations["sentiment_score"]
    )

    return recommendations.sort_values(
        "recommendation_score",
        ascending=False
    ).head(n)

# Generate Recommendations for NatWag Product


In [ ]:
recommendations = get_recommendations(target_index, n=5)

print(
    recommendations[
        [
            "title",
            "store",
            "review_count",
            "avg_review_rating",
            "positive_pct",
            "similarity_score",
            "recommendation_score"
        ]
    ].to_string(index=False)
)

                                                                                                                                                                                        title      store  review_count  avg_review_rating  positive_pct  similarity_score  recommendation_score
NATWAG Reusable Update Makeup Remover Cloths for Sensitive Skin - Microfiber Face Cloth Makeup Remover | Washable Round Makeup Remover Pads | Natural Facial Towels (2 Pads,2 Cloth) (Purple)     Natwag            17           4.823529     94.117647          0.416596              0.674092
                              A24 Cleansing Oil - Makeup Remover, Deep Cleansing, Non-Stripping, 99.75% Natural Ingredients, Vegan Formula, Ideal for All Skin Types… (Cleansing Oil to Foam)        A24            10           4.600000     80.000000          0.221755              0.537582
                                 Bioré J-Beauty Makeup Removing Moisturizing Cleansing Jelly, Top Japanese Makeup Remover, Gentle Skin C

# Analyze Similarity Score Distribution

In [ ]:
# Remove self-similarity scores from the diagonal
mask = ~np.eye(similarity_matrix.shape[0], dtype=bool)

similarity_values = similarity_matrix[mask]

print("Minimum similarity:", round(similarity_values.min(), 3))
print("Maximum similarity:", round(similarity_values.max(), 3))
print("Median similarity:", round(np.median(similarity_values), 3))
print("90th percentile:", round(np.percentile(similarity_values, 90), 3))
print("95th percentile:", round(np.percentile(similarity_values, 95), 3))

Minimum similarity: 0.0
Maximum similarity: 0.947
Median similarity: 0.0
90th percentile: 0.057
95th percentile: 0.091


how many candidates remain at 0.05,0.10,0.15, that will tell us whether 0.10 give us enough recommendations across the dataset

In [ ]:
for threshold in [0.05, 0.10, 0.15]:
    candidate_counts = []

    for i in range(len(product_analysis)):
        scores = similarity_matrix[i]
        candidate_counts.append((scores >= threshold).sum() - 1)

    print(
        f"Threshold {threshold:.2f}: "
        f"Average candidates = {np.mean(candidate_counts):.1f}, "
        f"Products with at least 5 candidates = {sum(c >= 5 for c in candidate_counts)}"
    )

Threshold 0.05: Average candidates = 28.0, Products with at least 5 candidates = 229
Threshold 0.10: Average candidates = 10.2, Products with at least 5 candidates = 170
Threshold 0.15: Average candidates = 4.9, Products with at least 5 candidates = 72


test the products

In [ ]:
test_products = product_analysis[
    product_analysis["review_count"] >= 10
].sample(5, random_state=42)[
    ["item_id", "title", "review_count", "avg_review_rating"]
]

test_products

,item_id,title,review_count,avg_review_rating
24,B07LCHCD6Q,"Water Flosser Cordless Teeth Cleaner, SOOCAS P...",10,4.000000
6,B015A5DGG4,12ML Refillable Perfume Atomizer Bottle for Tr...,11,4.454545
93,B088NVJSJT,DeepDream Portable Electric Nail Drill - Profe...,12,4.666667
109,B08BKWLHYP,ZOUSZ Black Oud Vegan Beard Balm - Classic Woo...,10,4.300000
104,B08B3SJNL5,"5pcs Nail Art Liner Brushes, Lystaii Dual-ende...",11,4.636364


the code above gives us a nice set of test cases to practice with:

**Why??** It covers several different product types:

1. water flosser
2. perfume atomizer
3. electric nail drill
4. beard balm
5. nail art brushes

Next step - covers generating recommendations for these 5

We'll also use the 0.10 similarity threshold we just justified and inspect the top 5 recommendations for each product

In [ ]:
for _, product in test_products.iterrows():
    product_index = product_analysis.index[
        product_analysis["item_id"] == product["item_id"]
    ][0]

    recommendations = get_recommendations(
        product_index,
        n=5,
        min_similarity=0.10
    )

    print("=" * 100)
    print(f"TARGET: {product['title']}")
    print(f"Average Rating: {product['avg_review_rating']:.2f}")
    print()
    print(
        recommendations[
            [
                "title",
                "review_count",
                "avg_review_rating",
                "positive_pct",
                "similarity_score",
                "recommendation_score"
            ]
        ].to_string(index=False)
    )

TARGET: Water Flosser Cordless Teeth Cleaner, SOOCAS Professional Oral Irrigator 230ML Portable and Rechargeable Water Dental Flosser with 3 Modes and 4 Jet Tips, IPX7 Waterproof, USB Charged for 30 Days Use
Average Rating: 4.00

                                                                                                                                                                                             title  review_count  avg_review_rating  positive_pct  similarity_score  recommendation_score
C'est Moi Gentle Cleansing Water | Gentle, Fragrance-Free Cleanser Removes Makeup and Refreshes Skin, Made with Cornflower Water, Aloe Vera, and Strawberry, Kiwi, and Apple Extracts | 5.9 fl oz.            12           4.083333          75.0          0.107966              0.454623
TARGET: 12ML Refillable Perfume Atomizer Bottle for Travel Spray Scent Pump Case Pack of 5
Average Rating: 4.45

                                                                                             

# Load Metadata

In [ ]:
metadata_url = "https://huggingface.co/datasets/debolut/amazon-reviews-2023-all-beauty-sample/resolve/main/meta_sample.parquet"

metadata = pd.read_parquet(metadata_url)

print("Metadata shape:", metadata.shape)
print("Available columns:", metadata.columns.tolist())

Metadata shape: (2216, 16)
Available columns: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'item_id', 'bought_together', 'subtitle', 'author']


# Inspect Metadata Quality

In [ ]:
#getting the fields
metadata_fields = [
    "categories",
    "features",
    "description",
    "details"
]

for column in metadata_fields:
    def has_content(value):
        if isinstance(value, (list, dict)):
            return len(value) > 0
        if isinstance(value, str):
            return bool(value.strip()) and value.strip() != "[]"
        return False

    non_empty = metadata[column].apply(has_content).sum()

    print(
        f"{column}: {non_empty:,} non-empty "
        f"({non_empty / len(metadata) * 100:.1f}%)"
    )

categories: 0 non-empty (0.0%)
features: 536 non-empty (24.2%)
description: 298 non-empty (13.4%)
details: 2,216 non-empty (100.0%)


In [ ]:
attribute_counts = Counter()

for value in metadata["details"]:
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except (json.JSONDecodeError, TypeError):
            continue

    if isinstance(value, dict):
        attribute_counts.update(value.keys())

pd.DataFrame(
    attribute_counts.most_common(20),
    columns=["attribute", "product_count"]
)


,attribute,product_count
0,Brand,1665
1,Package Dimensions,1364
2,UPC,1243
3,Item Form,832
4,Product Dimensions,676
5,Is Discontinued By Manufacturer,597
6,Manufacturer,577
7,Material,565
8,Unit Count,559
9,Item model number,500


#Build Richer Product Text

In [ ]:
# Convert the details field into dictionaries
def parse_details(value):
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, dict) else {}
        except (json.JSONDecodeError, TypeError):
            return {}

    return {}


metadata["parsed_details"] = metadata["details"].apply(parse_details)

# Select attributes that may describe product use or characteristics
useful_attributes = [
    "Item Form",
    "Material",
    "Hair Type",
    "Skin Type",
    "Scent",
    "Product Benefits",
    "Special Feature",
    "Finish Type"
]

def extract_product_attributes(details):
    return " ".join(
        str(details[key])
        for key in useful_attributes
        if key in details and details[key]
    )

metadata["selected_attributes"] = metadata["parsed_details"].apply(
    extract_product_attributes
)

# Merge the selected attributes into the product analysis table
product_analysis = product_analysis.drop(
    columns=["selected_attributes"],
    errors="ignore"
)

product_analysis = product_analysis.merge(
    metadata[["item_id", "selected_attributes"]],
    on="item_id",
    how="left"
)

product_analysis["selected_attributes"] = (
    product_analysis["selected_attributes"].fillna("")
)

product_analysis[
    ["title", "selected_attributes"]
].head()

,title,selected_attributes
0,Retinol Face Moisturizer Cream Natural Facial ...,Cream flower
1,OZNaturals Facial Mask - This Moisturizing Fac...,Dry Unscented Hydrating Not Tested On Animals
2,Argan Magic 100% Pure Argan Oil – Moisturizing...,Oil
3,Pure Hyaluronic Acid Serum with Vitamin C for ...,All Aloe Anti-Aging
4,"Nadira Organics Virgin Argan Oil for Skin, Fac...",Oil Fragrance Free


In [ ]:
product_analysis["enhanced_product_text"] = (
    product_analysis["title"].fillna("") + " " +
    product_analysis["selected_attributes"].fillna("")
).apply(clean_product_text)

product_analysis[
    ["title", "product_text", "enhanced_product_text"]
].head()

,title,product_text,enhanced_product_text
0,Retinol Face Moisturizer Cream Natural Facial ...,retinol face moisturizer cream natural facial ...,retinol face moisturizer cream natural facial ...
1,OZNaturals Facial Mask - This Moisturizing Fac...,oznaturals facial mask this moisturizing face ...,oznaturals facial mask this moisturizing face ...
2,Argan Magic 100% Pure Argan Oil – Moisturizing...,argan magic 100 pure argan oil moisturizing oi...,argan magic 100 pure argan oil moisturizing oi...
3,Pure Hyaluronic Acid Serum with Vitamin C for ...,pure hyaluronic acid serum with vitamin c for ...,pure hyaluronic acid serum with vitamin c for ...
4,"Nadira Organics Virgin Argan Oil for Skin, Fac...",nadira organics virgin argan oil for skin face...,nadira organics virgin argan oil for skin face...


# Build advanced tfidf matrix

In [ ]:
tfidf_enhanced = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_enhanced_matrix = tfidf_enhanced.fit_transform(
    product_analysis["enhanced_product_text"]
)

similarity_enhanced_matrix = cosine_similarity(
    tfidf_enhanced_matrix
)

print("Enhanced TF-IDF matrix shape:", tfidf_enhanced_matrix.shape)
print("Enhanced similarity matrix shape:", similarity_enhanced_matrix.shape)

Enhanced TF-IDF matrix shape: (239, 1726)
Enhanced similarity matrix shape: (239, 239)


# Compare the two models on the water flosser

In [ ]:
# Find the water flosser in the product analysis table
target_index = product_analysis.index[
    product_analysis["title"].str.contains(
        "SOOCAS Professional Oral Irrigator",
        case=False,
        na=False
    )
][0]

# Compare the top 5 similar products from both models
for model_name, matrix in [
    ("Baseline", similarity_matrix),
    ("Enhanced", similarity_enhanced_matrix)
]:
    scores = matrix[target_index]

    similar_indices = scores.argsort()[::-1]
    similar_indices = similar_indices[similar_indices != target_index]
    similar_indices = [
        i for i in similar_indices
        if scores[i] >= 0.10
    ][:5]

    print("=" * 90)
    print(f"{model_name} model — Water Flosser")
    print()

    if similar_indices:
        comparison = product_analysis.iloc[similar_indices][
            ["title", "review_count", "avg_review_rating"]
        ].copy()

        comparison["similarity_score"] = [
            scores[i] for i in similar_indices
        ]

        print(comparison.to_string(index=False))
    else:
        print("No products met the similarity threshold.")

Baseline model — Water Flosser

                                                                                                                                                                                             title  review_count  avg_review_rating  similarity_score
C'est Moi Gentle Cleansing Water | Gentle, Fragrance-Free Cleanser Removes Makeup and Refreshes Skin, Made with Cornflower Water, Aloe Vera, and Strawberry, Kiwi, and Apple Extracts | 5.9 fl oz.            12           4.083333          0.107966
Enhanced model — Water Flosser

                                                                                                                          title  review_count  avg_review_rating  similarity_score
20pcs Nail Tips Clips for Quick Building Polygel Nail Forms Nail Tips Clip for Finger Nail Extension DIY Manicure Nail Art Tool            12               4.25          0.105335


checking that the waterflosser has useful metadata

In [ ]:
# Inspect the water flosser's product information
water_flosser = product_analysis[
    product_analysis["title"].str.contains(
        "SOOCAS Professional Oral Irrigator",
        case=False,
        na=False
    )
]

water_flosser[
    ["title", "store", "selected_attributes", "enhanced_product_text"]
].T

,24
title,"Water Flosser Cordless Teeth Cleaner, SOOCAS P..."
store,SOOCAS
selected_attributes,"Por, Multiple Operation Modes, Multiple Tips"
enhanced_product_text,water flosser cordless teeth cleaner soocas pr...


# Word2Vec: Semantic Product Representations

## Description:

Word2Vec learns dense word embeddings from contextual relationships in text. We represent each product by averaging the embeddings of its words and use cosine similarity to identify semantically similar products.

Unlike the TF-IDF baseline, this approach can capture relationships between words based on their usage in the training corpus.

In [ ]:
#tokenizing the product text for Word2Vec
tokenized_products = (
    product_analysis["product_text"]
    .fillna("")
    .apply(lambda text: text.split())
    .tolist()
)

#preview the tokenized text
print(tokenized_products[0])

['retinol', 'face', 'moisturizer', 'cream', 'natural', 'facial', 'moisturizing', 'cream', '1', '76', 'oz', 'with', 'ortho', 'hydroxybenzoic', 'acid', 've', 'for', 'hydrating', 'repairing', 'instanatural']


In [ ]:
#training the Word2Vec model using the tokenized products
word2vec_model = Word2Vec(
    sentences=tokenized_products,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=100,
    seed=42
)

In [ ]:
#inspecting the trained Word2Vec model
print("Vocabulary size:", len(word2vec_model.wv))
print("Embedding dimensions:", word2vec_model.vector_size)

#inspect a sample word embedding
print("\nExample word embedding:")
print(word2vec_model.wv["moisturizer"])

Vocabulary size: 1841
Embedding dimensions: 100

Example word embedding:
[ 1.1378244e+00 -3.7401307e-01  3.4690008e-01 -7.4405098e-01
 -8.0199772e-01 -3.8473216e-01  1.8483682e-01  4.8745996e-01
 -1.0197911e+00 -8.4167218e-01 -1.0399083e+00 -1.1116800e-01
  3.7481236e-01  1.7231801e-01 -1.7645404e-01 -4.9248576e-01
  6.6150658e-02 -2.4526879e-02 -6.6206139e-01  5.3370267e-01
  1.8413952e-02 -1.4446811e-01  1.8839063e-01 -4.6320143e-01
  7.0236242e-01  9.4773835e-01 -3.8048974e-01  4.2994922e-01
  2.6179090e-01 -4.5320544e-01  6.6409983e-02  1.6066523e-01
  4.2148805e-01  7.0319498e-01 -3.4368294e-01 -6.0970134e-01
 -3.4700868e-01 -4.0819487e-01 -1.6698678e-01  3.7141272e-01
  8.7323064e-01  7.5460895e-04  7.2040790e-01 -3.0158645e-01
  3.3343308e-02 -7.0910841e-02  2.4660473e-01  1.3825534e-01
  3.9970592e-01 -1.5920146e-01 -1.0689178e+00 -4.6717364e-01
 -1.7774542e-01  3.2709834e-01 -5.8822173e-01 -1.2060515e+00
  7.2658598e-01  4.8863435e-01 -1.9716331e-01  2.8788486e-01
 -7.6269872e

# Word2Vec: Generate Product Embeddings

## Description:
Word2Vec generates numerical embeddings for individual words. To represent entire products, we calculate the average of the embeddings associated with each product's tokens.

This produces a 100 dimensional vector for each product, allowing us to compare products using cosine similarity

In [ ]:
#create the product embeddings --> generate an embedding for each product
def get_product_embedding(tokens, model):
  word_vectors = [
      model.wv[word]
      for word in tokens
      if word in tokens
  ]

  if not word_vectors:
    return np.zeros(model.vector_size)

  return np.mean(word_vectors, axis=0)

product_embeddings = np.vstack([
    get_product_embedding(tokens, word2vec_model)
    for tokens in tokenized_products
])

print("Product embedding matrix shape:", product_embeddings.shape)

Product embedding matrix shape: (239, 100)


# Calculate Word2Vec Cosine Similarity


In [ ]:
#calculate word2vec cosine similarity
# Calculate pairwise cosine similarity between product embeddings
word2vec_similarity_matrix = cosine_similarity(product_embeddings)

print("Word2Vec similarity matrix shape:", word2vec_similarity_matrix.shape)

Word2Vec similarity matrix shape: (239, 239)


In [ ]:
#examine Word2Vec similarity score distribution
print("Minimum similarity:", word2vec_similarity_matrix.min())
print("Maximum similarity:", word2vec_similarity_matrix.max())
print("Median similarity:", np.median(word2vec_similarity_matrix))
print("90th percentile:", np.percentile(word2vec_similarity_matrix, 90))
print("95th percentile:", np.percentile(word2vec_similarity_matrix, 95))

Minimum similarity: -0.22588466
Maximum similarity: 1.0000004
Median similarity: 0.50281894
90th percentile: 0.69341844
95th percentile: 0.7545544


In [ ]:
#building a word2vec recommendation function
def recommend_word2vec(product_name, top_n = 5):

  #find products matching the search term
  matches = product_analysis[
      product_analysis["title"].str.contains(
          product_name,
          case=False,
          na=False
      )
  ]

  if matches.empty:
    print("No matching product found.")
    return

  #get the position of the first matching product
  product_index = product_analysis.index.get_loc(matches.index[0])

  #retrieve similarity scores for the selected product
  similarity_scores = word2vec_similarity_matrix[product_index]

  #rank products by similarity excluding the selected product
  similar_indices = np.argsort(similarity_scores)[::-1][1:top_n + 1]

  #display recommended products
  recommendations = product_analysis.iloc[similar_indices][
      ["title", "store", "avg_review_rating", "positive_pct"]
  ].copy()

  recommendations["similarity_score"] = similarity_scores[similar_indices]

  return recommendations.reset_index(drop=True)

In [ ]:
#generate your first word2vec recommendations, and we can start by checking which products are available so we can choose a meaningful test product
product_analysis[["title", "store"]].head(10)

,title,store
0,Retinol Face Moisturizer Cream Natural Facial ...,InstaNatural
1,OZNaturals Facial Mask - This Moisturizing Fac...,Auperwel
2,Argan Magic 100% Pure Argan Oil – Moisturizing...,Argan Magic
3,Pure Hyaluronic Acid Serum with Vitamin C for ...,Derma-nu Miracle Skin Remedies
4,"Nadira Organics Virgin Argan Oil for Skin, Fac...",UNGLINGA
5,"Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths ...",Pure Body Naturals
6,12ML Refillable Perfume Atomizer Bottle for Tr...,Rioa
7,"Fungus Nail Treatment,Sky-shop Fungus Nail Rep...",Pei Mei
8,Spanature Olive Hand & Body Lotion Travel Size...,Spanature
9,"AuroTrends Makeup Remover Towel,2 Pack Reusabl...",AuroTrends


In [ ]:
# test word2vec recommendations
recommend_word2vec("Argan Oil", top_n=5)

,title,store,avg_review_rating,positive_pct,similarity_score
0,Argan Magic Color Protection Hair Mask – Nouri...,Argan Magic,4.416667,91.666667,0.918752
1,Organic Sweet Almond Oil and Fractionated Coco...,Shiny Leaf,4.200000,80.000000,0.883543
2,CELLEANSER Vegan Clean Cream 100% Vegan Ingred...,Celleanser,4.909091,100.000000,0.863369
3,Vegan Mia - USDA Organic Radiance Superfruit C...,Vegan Mia,4.136364,77.272727,0.858629
4,Elli K Essential Sincerity From AZ Time Revers...,ELLI K,4.076923,84.615385,0.843490


In [ ]:
#lets inspect the matching products --> check whether the problem is word2vec embeddings or simply limitaitons
product_analysis[
    product_analysis["title"].str.contains(
        "Argan Oil",
        case = False,
        na = False
    )
][["title", "store", "avg_review_rating", "positive_pct"]]

,title,store,avg_review_rating,positive_pct
2,Argan Magic 100% Pure Argan Oil – Moisturizing...,Argan Magic,3.866667,60.00
4,"Nadira Organics Virgin Argan Oil for Skin, Fac...",UNGLINGA,4.818182,100.00
5,"Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths ...",Pure Body Naturals,4.625000,93.75


In [ ]:
#comparing recommendation results, across the 3 tfidf
product_analysis.loc[
    product_analysis["title"].str.contains(
        "Argan Oil",
        case=False,
        na=False
    ),
    ["item_id", "title", "store"]
]

,item_id,title,store
2,B00O2FGBJS,Argan Magic 100% Pure Argan Oil – Moisturizing...,Argan Magic
4,B00OXDWFG2,"Nadira Organics Virgin Argan Oil for Skin, Fac...",UNGLINGA
5,B012Q9NGE4,"Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths ...",Pure Body Naturals


# Testing the Word2Vec with the TFIDF

In [ ]:
#select the reference product for model comparison
target_product_id = "B00O2FGBJS"

target_index = product_analysis.index.get_loc(
    product_analysis[
        product_analysis["item_id"] == target_product_id
    ].index[0]
)

print("Reference product:")
print(product_analysis.iloc[target_index]["title"])
print("Product index:", target_index)

Reference product:
Argan Magic 100% Pure Argan Oil – Moisturizing Oil that Infuses Skin with Enriched Hydration | Leaves Skin Restored & Refreshed | Non-Greasy | Rich in Vitamin E | Made in USA | Cruelty Free (2 oz)
Product index: 2


In [ ]:
#generate original tfidf recommendations

# retrieve original TF-IDF similarity scores
tfidf_scores = similarity_matrix[target_index]

# rank products by similarity, excluding the reference product
tfidf_indices = np.argsort(tfidf_scores)[::-1][1:6]

# display the top five recommendations
tfidf_recommendations = product_analysis.iloc[tfidf_indices][
    ["title", "store", "avg_review_rating", "positive_pct"]
].copy()

tfidf_recommendations["similarity_score"] = tfidf_scores[tfidf_indices]

tfidf_recommendations.reset_index(drop=True)

,title,store,avg_review_rating,positive_pct,similarity_score
0,Argan Magic Color Protection Hair Mask – Nouri...,Argan Magic,4.416667,91.666667,0.472415
1,"Nadira Organics Virgin Argan Oil for Skin, Fac...",UNGLINGA,4.818182,100.000000,0.248254
2,Organic Sweet Almond Oil and Fractionated Coco...,Shiny Leaf,4.200000,80.000000,0.225537
3,"Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths ...",Pure Body Naturals,4.625000,93.750000,0.205220
4,100% PURE Coconut Body Scrub Exfoliating Deep ...,100% PURE,4.300000,80.000000,0.145304


In [ ]:
#testing with the enhanced tf-idf recommendations
# Retrieve enhanced TF-IDF similarity scores
enhanced_scores = similarity_enhanced_matrix[target_index]

# Rank products by similarity, excluding the reference product
enhanced_indices = np.argsort(enhanced_scores)[::-1][1:6]

# Display the top five recommendations
enhanced_recommendations = product_analysis.iloc[enhanced_indices][
    ["title", "store", "avg_review_rating", "positive_pct"]
].copy()

enhanced_recommendations["similarity_score"] = enhanced_scores[enhanced_indices]

enhanced_recommendations.reset_index(drop=True)

,title,store,avg_review_rating,positive_pct,similarity_score
0,"Nadira Organics Virgin Argan Oil for Skin, Fac...",UNGLINGA,4.818182,100.000000,0.331110
1,Organic Sweet Almond Oil and Fractionated Coco...,Shiny Leaf,4.200000,80.000000,0.315336
2,Argan Magic Color Protection Hair Mask – Nouri...,Argan Magic,4.416667,91.666667,0.284724
3,"Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths ...",Pure Body Naturals,4.625000,93.750000,0.259153
4,Vegan Mia - USDA Organic Radiance Superfruit C...,Vegan Mia,4.136364,77.272727,0.215179


# Word2Vec: Comparing Product Similarity Methods

## Description:

**We compare three approaches to product recommendations using the same reference product. These comparisons helps evaluate how different text representations influence product recommendations.**

1. Original TF-IDF: Keyword-based similarity using product titles and store names.

2. Enhanced TF-IDF: Keyword-based similarity incorporating additional product attributes.

3. Word2Vec: Semantic similarity using averaged word embeddings.

In [ ]:
# Compare top recommendations across all three methods
comparison = pd.DataFrame({
    "Original TF-IDF": tfidf_recommendations["title"].tolist(),
    "Enhanced TF-IDF": enhanced_recommendations["title"].tolist(),
    "Word2Vec": recommend_word2vec(
        "Argan Magic 100% Pure Argan Oil",
        top_n=5
    )["title"].tolist()
})

comparison

,Original TF-IDF,Enhanced TF-IDF,Word2Vec
0,Argan Magic Color Protection Hair Mask – Nouri...,"Nadira Organics Virgin Argan Oil for Skin, Fac...",Argan Magic Color Protection Hair Mask – Nouri...
1,"Nadira Organics Virgin Argan Oil for Skin, Fac...",Organic Sweet Almond Oil and Fractionated Coco...,Organic Sweet Almond Oil and Fractionated Coco...
2,Organic Sweet Almond Oil and Fractionated Coco...,Argan Magic Color Protection Hair Mask – Nouri...,CELLEANSER Vegan Clean Cream 100% Vegan Ingred...
3,"Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths ...","Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths ...",Vegan Mia - USDA Organic Radiance Superfruit C...
4,100% PURE Coconut Body Scrub Exfoliating Deep ...,Vegan Mia - USDA Organic Radiance Superfruit C...,Elli K Essential Sincerity From AZ Time Revers...


In [ ]:
#lets test other products to make sure our model is working

#examine product category distribution
product_analysis["main_category"].value_counts().head(15)

# Display product titles for evaluation-set selection
for i, title in enumerate(product_analysis["title"].head(30), start=1):
    print(f"{i}. {title}")

1. Retinol Face Moisturizer Cream Natural Facial moisturizing Cream 1.76 OZ with Ortho-Hydroxybenzoic Acid,VE for Hydrating,Repairing
2. OZNaturals Facial Mask - This Moisturizing Face Mask Contains Rosehip Oil, Pro Vitamin B5 & Amino Acids - This Anti Aging Mask Deeply Hydrates Dry Skin For A Healthy Youthful Glow!
3. Argan Magic 100% Pure Argan Oil – Moisturizing Oil that Infuses Skin with Enriched Hydration | Leaves Skin Restored & Refreshed | Non-Greasy | Rich in Vitamin E | Made in USA | Cruelty Free (2 oz)
4. Pure Hyaluronic Acid Serum with Vitamin C for Face - Organic Anti Aging & Anti-Wrinkle Skin Treatment - Penetrates Deeply & Activates Collagen Production for a Younger, Firmer & Brighter Complexion
5. Nadira Organics Virgin Argan Oil for Skin, Face, Hair, and Nails, 4 fl. oz.
6. Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths and Repairs - Sulfate Free - Natural - Imported from Morocco by Pure Body Naturals
7. 12ML Refillable Perfume Atomizer Bottle for Travel Spray Scent Pump

In [ ]:
#create evaluation set - Let's store the five product IDs so we can reuse the exact same reference products across all three recommendation methods.
# Define five reference products for model evaluation
evaluation_product_ids = [
    "B00O2FGBJS",  # Argan Magic Argan Oil
]

print("Evaluation products:", evaluation_product_ids)

Evaluation products: ['B00O2FGBJS']


In [ ]:
#retrieve product ids

#find the four additional evaluation products
evaluation_keywords = [
    "Retinol Face Moisturizer",
    "Argan Magic Color Protection Hair Mask",
    "Makeup Remover Towel",
    "Water Flosser Cordless Teeth Cleaner"
]

for keyword in evaluation_keywords:
    matches = product_analysis[
        product_analysis["title"].str.contains(
            keyword,
            case=False,
            na=False
        )
    ]

    print(f"\n{keyword}")

    if matches.empty:
        print("No matching product found.")
    else:
        print(matches[["item_id", "title"]].to_string(index=False))


Retinol Face Moisturizer
   item_id                                                                                                                              title
B00KCTER3U Retinol Face Moisturizer Cream Natural Facial moisturizing Cream 1.76 OZ with Ortho-Hydroxybenzoic Acid,VE for Hydrating,Repairing

Argan Magic Color Protection Hair Mask
   item_id                                                                                                                                                                                                  title
B07L9H27SH Argan Magic Color Protection Hair Mask – Nourishes, Hydrates & Helps to Protect Color Treated Hair | Enriched with Moringa Oil & Botanical Extracts | Made in USA | Paraben Free | Cruelty Free (8 oz)

Makeup Remover Towel
   item_id                                                                                                                                                                                title
B071R2QPF3 AuroT

In [ ]:
#complete the evaluaton set
# Define five reference products for model evaluation
evaluation_product_ids = [
    "B00O2FGBJS",  # Argan Magic Argan Oil
    "B00KCTER3U",  # Retinol Face Moisturizer
    "B07L9H27SH",  # Argan Magic Hair Mask
    "B071R2QPF3",  # Makeup Remover Towel
    "B07LCHCD6Q"   # Water Flosser
]

# Display evaluation products
evaluation_products = product_analysis[
    product_analysis["item_id"].isin(evaluation_product_ids)
][["item_id", "title", "store"]]

print("Number of evaluation products:", len(evaluation_products))
display(evaluation_products)

Number of evaluation products: 5


,item_id,title,store
0,B00KCTER3U,Retinol Face Moisturizer Cream Natural Facial ...,InstaNatural
2,B00O2FGBJS,Argan Magic 100% Pure Argan Oil – Moisturizing...,Argan Magic
9,B071R2QPF3,"AuroTrends Makeup Remover Towel,2 Pack Reusabl...",AuroTrends
23,B07L9H27SH,Argan Magic Color Protection Hair Mask – Nouri...,Argan Magic
24,B07LCHCD6Q,"Water Flosser Cordless Teeth Cleaner, SOOCAS P...",SOOCAS


# Word2Vec: Multi-Product Recommendation Evaluation

## Description:

To evaluate recommendation consistency across different product types, we test five reference products using the same recommendation procedure.

Each similarity method returns its top five recommendations for each reference product. This allows us to compare recommendation relevance across multiple inputs rather than relying on a single example.

In [ ]:
def evaluate_recommendations(similarity_matrix, product_id, top_n=5):
  #find the position of the reference product
  product_index = np.flatnonzero(
      product_analysis["item_id"].eq(product_id).to_numpy()
  )[0]

  #retrieve similarity scores
  similarity_scores = similarity_matrix[product_index]

  #rank products by similarity, excluding the reference product
  similar_indices = np.argsort(similarity_scores)[::-1][1:top_n + 1]

  #create recommendations
  recommendations = product_analysis.iloc[similar_indices][
      ["item_id", "title", "store", "avg_review_rating", "positive_pct"]
  ].copy()

  recommendations["similarity_score"] = similarity_scores[similar_indices]

  return recommendations.reset_index(drop=True)

In [ ]:
#test the evaluation function with the Argan Oil reference product
test_recommendations = evaluate_recommendations(
    word2vec_similarity_matrix,
    "B00O2FGBJS",
    top_n=5
)

display(test_recommendations)

,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B07L9H27SH,Argan Magic Color Protection Hair Mask – Nouri...,Argan Magic,4.416667,91.666667,0.918752
1,B08LYT4Q2X,Organic Sweet Almond Oil and Fractionated Coco...,Shiny Leaf,4.200000,80.000000,0.883543
2,B0B5YVX9YY,CELLEANSER Vegan Clean Cream 100% Vegan Ingred...,Celleanser,4.909091,100.000000,0.863369
3,B0BR4GL7Q6,Vegan Mia - USDA Organic Radiance Superfruit C...,Vegan Mia,4.136364,77.272727,0.858629
4,B0B4JPGX8P,Elli K Essential Sincerity From AZ Time Revers...,ELLI K,4.076923,84.615385,0.843490


In [ ]:
#evaluate Word2Vec recommendations across all five reference products
for product_id in evaluation_product_ids:

  product_title = product_analysis.loc[
      product_analysis["item_id"] == product_id, "title"
  ].iloc[0]

  print(f"\nReference Product: {product_title}")

  recommendations = evaluate_recommendations(
      word2vec_similarity_matrix,
      product_id,
      top_n = 5
  )

  display(recommendations)


Reference Product: Argan Magic 100% Pure Argan Oil – Moisturizing Oil that Infuses Skin with Enriched Hydration | Leaves Skin Restored & Refreshed | Non-Greasy | Rich in Vitamin E | Made in USA | Cruelty Free (2 oz)


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B07L9H27SH,Argan Magic Color Protection Hair Mask – Nouri...,Argan Magic,4.416667,91.666667,0.918752
1,B08LYT4Q2X,Organic Sweet Almond Oil and Fractionated Coco...,Shiny Leaf,4.200000,80.000000,0.883543
2,B0B5YVX9YY,CELLEANSER Vegan Clean Cream 100% Vegan Ingred...,Celleanser,4.909091,100.000000,0.863369
3,B0BR4GL7Q6,Vegan Mia - USDA Organic Radiance Superfruit C...,Vegan Mia,4.136364,77.272727,0.858629
4,B0B4JPGX8P,Elli K Essential Sincerity From AZ Time Revers...,ELLI K,4.076923,84.615385,0.843490



Reference Product: Retinol Face Moisturizer Cream Natural Facial moisturizing Cream 1.76 OZ with Ortho-Hydroxybenzoic Acid,VE for Hydrating,Repairing


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B08HVRP54L,So'Bio Étic | Moisturizing Fresh Gel Cream | O...,So'Bio Étic,4.100000,60.000000,0.902090
1,B08SQZ1B3W,Murad Hydration Intense Recovery Cream - Deepl...,NaN,4.500000,83.333333,0.859727
2,B08GJJ5RV9,"Facial Skincare,Cucumber Essence Hydrating Moi...",PalNour,4.400000,80.000000,0.855074
3,B08NJ5BTWG,FASCY LAB GREEN+ Korean Sunscreen — SPF 50+PA+...,Fascy,4.923077,100.000000,0.843000
4,B0855L611L,Lesentia Tinted Moisturizer with SPF 31 – Tint...,L'esentia,3.933333,66.666667,0.823774



Reference Product: Argan Magic Color Protection Hair Mask – Nourishes, Hydrates & Helps to Protect Color Treated Hair | Enriched with Moringa Oil & Botanical Extracts | Made in USA | Paraben Free | Cruelty Free (8 oz)


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B00O2FGBJS,Argan Magic 100% Pure Argan Oil – Moisturizing...,Argan Magic,3.866667,60.000000,0.918752
1,B08WCG4JCC,"GIOVANNI Hemp Hydrating Hair Shine Spray, 4.3 ...",GIOVANNI,4.500000,90.000000,0.845154
2,B08LYT4Q2X,Organic Sweet Almond Oil and Fractionated Coco...,Shiny Leaf,4.200000,80.000000,0.803671
3,B07THLR7RR,LANGE HAIR Extendé Conditioning Detangler Hair...,NaN,4.166667,66.666667,0.789415
4,B07FP2C8N8,Emilia Hypoallergenic Bath Oil for Babies - Pa...,Emilia,4.454545,81.818182,0.784926



Reference Product: AuroTrends Makeup Remover Towel,2 Pack Reusable Chemical-free Microfiber Makeup Remover Cloths,Magically and Gently Wipe Away Cosmetics,Sunscreen,Oil and Dirt,Set of 2 (Pink+Black)


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B088LWRYJC,NATWAG Reusable Update Makeup Remover Cloths f...,Natwag,4.823529,94.117647,0.900315
1,B087F14JSH,"Natwag Makeup Remover Pads,18 Pack 3.74 Inch R...",Natwag,4.777778,100.000000,0.823228
2,B09QT8SLJB,AOA Studio Collection Makeup Sponge Set Makeup...,AOA STUDIO,4.636364,81.818182,0.811843
3,B08XJWLLKQ,John Frieda Detox and Repair Shampoo and Condi...,John Frieda,4.454545,100.000000,0.763166
4,B09TTC9Q9W,mold remover gel (3 Pack),Yondarli,3.000000,41.666667,0.759593



Reference Product: Water Flosser Cordless Teeth Cleaner, SOOCAS Professional Oral Irrigator 230ML Portable and Rechargeable Water Dental Flosser with 3 Modes and 4 Jet Tips, IPX7 Waterproof, USB Charged for 30 Days Use


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B09G22586Y,"Automatic Curling Iron, Rechargeable Cordless ...",LUZZAM,4.000000,70.000000,0.792397
1,B08NPBQR9L,"3 Barrel Curling Iron, Milantia 3/4 Inch Mini ...",Milantia,4.142857,85.714286,0.777959
2,B07DY1RDQR,Replacement Toothbrush Heads Compatible With O...,HDY,4.200000,80.000000,0.719672
3,B09571TNVG,AORAEM Mini Nail Dust Collector Portable Sucti...,AORAEM,3.090909,36.363636,0.701149
4,B08BLX72TP,LeiCare Tweezers Precision 2-Pack - Stainless ...,LeiCare,4.545455,90.909091,0.699649


# Word2Vec: Word2Vec Evaluation Findings

## Description:

**Word2Vec demonstrated varying recommendation quality across the five reference products:**

- **Strengths**: the model identified semantically related products, particularly for makeup removal, skincare, and haircare.
- **Limitations**: recommendations were less relevant for certain products, particularly the water flosser, where unrelated beauty and electronic devices appeared.
- **Similarity interpretations**: High cosine similarity scores did not always correspond to meaningful product relevance.


Overall, Word2Vec demonstrated the potential of semantic embeddings for product recommendations, but its performance was inconsistent. The relatively small training corpus may limit its ability to learn reliable product representations.

# GloVe: Downloading Pre-Trained Embeddings

In [ ]:
# Download pretrained GloVe embeddings from Hugging Face
glove_url = "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip"

glove_zip_path = "glove.6B.zip"

if not os.path.exists(glove_zip_path):
    urllib.request.urlretrieve(glove_url, glove_zip_path)

print("GloVe download complete.")

GloVe download complete.


In [ ]:
#extract only the 50-dimensional GloVe embeddings
with zipfile.ZipFile(glove_zip_path, "r") as zip_ref:
  zip_ref.extract("glove.6B.50d.txt", ".")
print("GloVe 50-dimensional embeddings extracted successfully.")

GloVe 50-dimensional embeddings extracted successfully.


In [ ]:
#Load GloVe Embeddings

#create a dictionary called glove_embeddings
glove_embeddings = {}

#using each word as a dictionary key
with open("glove.6B.50d.txt", "r", encoding="utf-8") as file:
  #storing its corresponding 50-dimensional numerical vector
  for line in file:
    values = line.split()
    word = values[0]
    vector = np.asarray(values[1:], dtype="float32")
    glove_embeddings[word] = vector

#checking the vocab size and embedding dimensions
print("Vocabulary size:", len(glove_embeddings))
print("Embedding dimensions:", len(glove_embeddings["computer"]))

Vocabulary size: 400001
Embedding dimensions: 50


In [ ]:
#inspect the GloVe embedding for "moisturizer" as an example
print("Word: moisturizer")
print("Embedding dimensions:", len(glove_embeddings["moisturizer"]))
print("\nEmbedding vector:")
print(glove_embeddings["moisturizer"])

Word: moisturizer
Embedding dimensions: 50

Embedding vector:
[-0.2719    0.44403  -0.6118   -0.022197  0.10884   0.3895    0.9847
 -0.37259   0.44368   0.91498   1.1291    0.016042  1.0718    0.38862
 -0.056735  0.30883  -1.2126    0.065455  0.66807  -0.3647   -0.50267
  0.71615   0.26455  -0.48585  -0.51118   0.24781   0.011574  1.1982
  1.0791   -0.39923  -0.074704  0.79152  -0.31467   0.61114  -0.75771
  0.35601  -0.13793   0.69164   0.96589  -0.55612   1.498     0.11262
  0.7397    0.27967   0.33113  -0.55725   0.62432   0.28277   0.015991
  0.20294 ]


# GloVe: Create GloVe product embeddings

## Description

Similar to how we did the word2vec:

- GloVe will give 50-dimensional vector for each product

In [ ]:
#generate product embeddings using pre-trained GloVe vectors
def get_glove_embedding(tokens, embeddings):

  word_vectors =[
      embeddings[word]
      for word in tokens
      if word in embeddings
  ]

  if not word_vectors:
    return np.zeros(50)

  return np.mean(word_vectors, axis=0)

glove_product_embeddings = np.vstack([
    get_glove_embedding(tokens, glove_embeddings)
    for tokens in tokenized_products
])

print("GloVe product embedding matrix shape: ", glove_product_embeddings.shape)


GloVe product embedding matrix shape:  (239, 50)


# GloVe: Calculate GloVe Cosine Similarity

##Description:


In [ ]:
#calculate cosine similarity between GloVe product embeddings
glove_similarity_matrix = cosine_similarity(glove_product_embeddings)

print("GloVe similarity matrix shape:",glove_similarity_matrix.shape)

GloVe similarity matrix shape: (239, 239)


In [ ]:
#examine the GloVe similarity score distributions
print("Minimum similarity:", glove_similarity_matrix.min())
print("Maximum similarity:", glove_similarity_matrix.max())
print("Median similarity:", np.median(glove_similarity_matrix))
print("90th percentile:", np.percentile(glove_similarity_matrix, 90))
print("95th percentile:", np.percentile(glove_similarity_matrix, 95))

Minimum similarity: -0.20346182184158298
Maximum similarity: 1.0000000000000007
Median similarity: 0.7955060449507869
90th percentile: 0.8890219849072264
95th percentile: 0.9080782930627854


# GloVe: Test GloVe Recommendations

##Description

In [ ]:
#Test GloVe recommendations using Argan Magic Argan Oil
glove_test_recommendations = evaluate_recommendations(
    glove_similarity_matrix,
    "B00O2FGBJS",
    top_n=5
)

display(glove_test_recommendations)

,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B07L9H27SH,Argan Magic Color Protection Hair Mask – Nouri...,Argan Magic,4.416667,91.666667,0.965642
1,B08KHRF9NY,Calendula Soap With Coconut Oil Soap - Natural...,Pure Extracts,4.300000,80.000000,0.959095
2,B08LYT4Q2X,Organic Sweet Almond Oil and Fractionated Coco...,Shiny Leaf,4.200000,80.000000,0.956353
3,B076DBLG1K,Natural Solution Himalayan Pink Salt Body Scru...,Natural Solution,4.416667,83.333333,0.950698
4,B012Q9NGE4,"Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths ...",Pure Body Naturals,4.625000,93.750000,0.944238


In [ ]:
#evaluate GloVe recommendations across all five reference products
for product_id in evaluation_product_ids:
  product_title = product_analysis.loc[
      product_analysis["item_id"] == product_id, "title"
  ].iloc[0]

  print(f"Reference Product: {product_title}")

  recommendations = evaluate_recommendations(
      glove_similarity_matrix,
      product_id,
      top_n=5
  )

  display(recommendations)

Reference Product: Argan Magic 100% Pure Argan Oil – Moisturizing Oil that Infuses Skin with Enriched Hydration | Leaves Skin Restored & Refreshed | Non-Greasy | Rich in Vitamin E | Made in USA | Cruelty Free (2 oz)


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B07L9H27SH,Argan Magic Color Protection Hair Mask – Nouri...,Argan Magic,4.416667,91.666667,0.965642
1,B08KHRF9NY,Calendula Soap With Coconut Oil Soap - Natural...,Pure Extracts,4.300000,80.000000,0.959095
2,B08LYT4Q2X,Organic Sweet Almond Oil and Fractionated Coco...,Shiny Leaf,4.200000,80.000000,0.956353
3,B076DBLG1K,Natural Solution Himalayan Pink Salt Body Scru...,Natural Solution,4.416667,83.333333,0.950698
4,B012Q9NGE4,"Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths ...",Pure Body Naturals,4.625000,93.750000,0.944238


Reference Product: Retinol Face Moisturizer Cream Natural Facial moisturizing Cream 1.76 OZ with Ortho-Hydroxybenzoic Acid,VE for Hydrating,Repairing


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B08HVRP54L,So'Bio Étic | Moisturizing Fresh Gel Cream | O...,So'Bio Étic,4.100000,60.000000,0.932560
1,B08GJJ5RV9,"Facial Skincare,Cucumber Essence Hydrating Moi...",PalNour,4.400000,80.000000,0.929842
2,B0855L611L,Lesentia Tinted Moisturizer with SPF 31 – Tint...,L'esentia,3.933333,66.666667,0.918025
3,B0B8DZ7H5F,"Frankies Bikinis Lip Butter Gloss, Sleeping Ve...",Frankies Bikinis,4.545455,100.000000,0.910070
4,B0851QJPZY,Gentlehomme Face Moisturizer For Men - Mens Fa...,Gentlehomme,4.100000,90.000000,0.908948


Reference Product: Argan Magic Color Protection Hair Mask – Nourishes, Hydrates & Helps to Protect Color Treated Hair | Enriched with Moringa Oil & Botanical Extracts | Made in USA | Paraben Free | Cruelty Free (8 oz)


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B00O2FGBJS,Argan Magic 100% Pure Argan Oil – Moisturizing...,Argan Magic,3.866667,60.000000,0.965642
1,B076DBLG1K,Natural Solution Himalayan Pink Salt Body Scru...,Natural Solution,4.416667,83.333333,0.938764
2,B08KHRF9NY,Calendula Soap With Coconut Oil Soap - Natural...,Pure Extracts,4.300000,80.000000,0.937420
3,B08LYT4Q2X,Organic Sweet Almond Oil and Fractionated Coco...,Shiny Leaf,4.200000,80.000000,0.936644
4,B0BR4GL7Q6,Vegan Mia - USDA Organic Radiance Superfruit C...,Vegan Mia,4.136364,77.272727,0.933948


Reference Product: AuroTrends Makeup Remover Towel,2 Pack Reusable Chemical-free Microfiber Makeup Remover Cloths,Magically and Gently Wipe Away Cosmetics,Sunscreen,Oil and Dirt,Set of 2 (Pink+Black)


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B08P15TB2V,Gel Nail Polish Set - TOEDO Fall Winter Nail G...,TOEDO,4.181818,81.818182,0.940158
1,B085SY4WC3,Pine Gel Nail Polish Set Chameleon Shimmering ...,pine,4.846154,100.000000,0.939739
2,B087F14JSH,"Natwag Makeup Remover Pads,18 Pack 3.74 Inch R...",Natwag,4.777778,100.000000,0.938883
3,B08GKVYS1Y,"JIADEL Gel Nail Polish Colors Set,Confetti Gli...",JIADEL,4.416667,83.333333,0.933440
4,B088LWRYJC,NATWAG Reusable Update Makeup Remover Cloths f...,Natwag,4.823529,94.117647,0.929205


Reference Product: Water Flosser Cordless Teeth Cleaner, SOOCAS Professional Oral Irrigator 230ML Portable and Rechargeable Water Dental Flosser with 3 Modes and 4 Jet Tips, IPX7 Waterproof, USB Charged for 30 Days Use


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B08L6GXY2S,UV LED Nail Lamp – 128W Rapid-Curing Gel Nails...,OVLUX,4.733333,100.000000,0.934961
1,B01M24DUXH,Spanature Olive Hand & Body Lotion Travel Size...,Spanature,4.545455,100.000000,0.919311
2,B087J3H22J,NEW Challenger Men's Rinse Free Hand Wash - 1o...,Challenger,4.333333,91.666667,0.917702
3,B08LPGZMQK,Spanature Aloe Vera Hand & Body Lotion Travel ...,Spanature,4.500000,91.666667,0.915600
4,B086GST51S,3D Mink Eyelashes Natural Long Lashes 2 in 1 S...,PURELEOR,4.421053,89.473684,0.907882


# GloVe: GloVe Evaluation Findings

**GloVe demonstrated the ability to identify broad semantic relationships between products, but recommendatoins relevance varied across the five reference products**

1. **Strengths**: GloVe identified related oil-based products, hair care products, and moisturizers, particularly for the argan oil reference products
2. **Limitations**: recommendations included unrelated products, especially for the makeup remover towel and water flosser reference products
3. **Similarity interpretations**: GloVe produced consistently high cosine similarity scores but high scores did not necessarily indicate meaningful product relevance

**GloVe captured broad semantics but struggled to distinguis with related vocabulary and products with similar functions**

# BERT: Downloading Necessary Tools

In [ ]:
#install Sentence Transformers
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 8.0 MB/s  0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


# BERT: Import Sentence Transformer

- **note**: The model will learn nothing new from our dataset during this step; we're using its existing pretrained knowledge

In [ ]:
#import sentence transformer

#SentenceTransformer gives access to pretrained models
from sentence_transformers import SentenceTransformer

#load pretrained Sentence-BERT mode

#all-MiniLM-L6-v2 is the model were using
bert_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Sentence-BERT model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-BERT model loaded successfully.


# BERT: Generate Sentence-BERT product embeddings

## Description:

The key difference is that Sentence-BERT generates contextual representations of complete product descriptions. This gives us a different approach to semantic similarity.

In [ ]:
#generate Sentence-BERT product embeddings
bert_product_embeddings = bert_model.encode(
    product_analysis["product_text"].fillna("").tolist(),
    show_progress_bar=True
)

print("Sentence-BERT embedding matrix shape: ", bert_product_embeddings.shape)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Sentence-BERT embedding matrix shape:  (239, 384)


# BERT: Calculate Sentence-BERT Cosine Similarity

##Description:

- Lower median (0.274): Sentence-BERT generally assigns lower similarity scores across your product catalog than Word2Vec and GloVe.

- More differentiation: Its scores suggest it distinguishes more clearly between products rather than assigning broadly high similarities.

- Maximum near 1.0: This is expected because each product is perfectly similar to itself.

- Negative minimum: Some product pairs have embeddings pointing in opposite directions in the vector space.


In [ ]:
#calculate cosine similarity between Sentence-BERT product embeddings
bert_similarity_matrix = cosine_similarity(bert_product_embeddings)
print("Sentence-BERT similarity matrix shape: ", bert_similarity_matrix)

Sentence-BERT similarity matrix shape:  [[1.0000001  0.5562149  0.49780834 ... 0.37670606 0.34123862 0.16882841]
 [0.5562149  0.99999994 0.53383714 ... 0.27499565 0.2846287  0.23686051]
 [0.49780834 0.53383714 0.99999976 ... 0.3283502  0.34628326 0.29157767]
 ...
 [0.37670606 0.27499565 0.3283502  ... 1.0000001  0.26969206 0.22580242]
 [0.34123862 0.2846287  0.34628326 ... 0.26969206 1.0000002  0.24344836]
 [0.16882841 0.23686051 0.29157767 ... 0.22580242 0.24344836 1.        ]]


##BERT: Similarity Score Distribution - Cosine Similarity

In [ ]:
#examine Sentence-BERT similarity score distribution
print("Minimum similarity:", bert_similarity_matrix.min())
print("Maximum similarity:", bert_similarity_matrix.max())
print("Median similarity:", np.median(bert_similarity_matrix))
print("90th percentile:", np.percentile(bert_similarity_matrix, 90))
print("95th percentile:", np.percentile(bert_similarity_matrix, 95))

Minimum similarity: -0.10758061
Maximum similarity: 1.0000005
Median similarity: 0.27357677
90th percentile: 0.45091516
95th percentile: 0.50938046


# BERT: Test Sentence-BERT with Argan Oil

## What's happening in this cell:

- The first three recommendations are strongly related through argan oil, hair care, and product ingredients.

- The final two recommendations are less directly related, although they share broader skincare and moisturizing characteristics.

- Sentence-BERT appears to recognize relationships beyond exact keyword overlap, which is one of the benefits of contextual sentence embeddings.

In [64]:
#evaluate Sentence-BERT recommendations for Argan Magic Argain Oil
bert_argan_recommendations = evaluate_recommendations(
    bert_similarity_matrix,
    "B00O2FGBJS",
    top_n=5
)

display(bert_argan_recommendations)

,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B07L9H27SH,Argan Magic Color Protection Hair Mask – Nouri...,Argan Magic,4.416667,91.666667,0.679818
1,B00OXDWFG2,"Nadira Organics Virgin Argan Oil for Skin, Fac...",UNGLINGA,4.818182,100.000000,0.649890
2,B012Q9NGE4,"Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths ...",Pure Body Naturals,4.625000,93.750000,0.631473
3,B096YHL82K,Gryph & IvyRose Coat of Arms Hydrating Lotion ...,Gryph & IvyRose,4.700000,100.000000,0.560392
4,B07DFNPVSF,Dr. Denese SkinScience Essential Lipid Anti Ag...,Dr. Denese,4.461538,84.615385,0.552890


## BERT: Evaluate remaining 4 products


In [65]:
#evaluate Sentence-BERT across all five reference products
for product_id in evaluation_product_ids:
  product_title = product_analysis.loc[
      product_analysis["item_id"] == product_id, "title"
  ].iloc[0]

  print(f"\nReference Product: {product_title}")

  recommendations = evaluate_recommendations(
      bert_similarity_matrix,
      product_id,
      top_n = 5
  )

  display(recommendations)


Reference Product: Argan Magic 100% Pure Argan Oil – Moisturizing Oil that Infuses Skin with Enriched Hydration | Leaves Skin Restored & Refreshed | Non-Greasy | Rich in Vitamin E | Made in USA | Cruelty Free (2 oz)


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B07L9H27SH,Argan Magic Color Protection Hair Mask – Nouri...,Argan Magic,4.416667,91.666667,0.679818
1,B00OXDWFG2,"Nadira Organics Virgin Argan Oil for Skin, Fac...",UNGLINGA,4.818182,100.000000,0.649890
2,B012Q9NGE4,"Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths ...",Pure Body Naturals,4.625000,93.750000,0.631473
3,B096YHL82K,Gryph & IvyRose Coat of Arms Hydrating Lotion ...,Gryph & IvyRose,4.700000,100.000000,0.560392
4,B07DFNPVSF,Dr. Denese SkinScience Essential Lipid Anti Ag...,Dr. Denese,4.461538,84.615385,0.552890



Reference Product: Retinol Face Moisturizer Cream Natural Facial moisturizing Cream 1.76 OZ with Ortho-Hydroxybenzoic Acid,VE for Hydrating,Repairing


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B07NPCT6L5,Dr. Au Anti-Aging Face Oil Retinol Serum by Au...,AÚ NATURAL SKINFOOD,4.400000,90.000000,0.617483
1,B07MN1KJ15,"MBA Renewing Retinol Serum, 1.7 fl. oz, 0.5% R...",Higher Education,3.833333,58.333333,0.613408
2,B08SQZ1B3W,Murad Hydration Intense Recovery Cream - Deepl...,NaN,4.500000,83.333333,0.583403
3,B0855L611L,Lesentia Tinted Moisturizer with SPF 31 – Tint...,L'esentia,3.933333,66.666667,0.565348
4,B08TR1MKHD,O!GETi Vitamin C Foam Cleanser | Foaming Facia...,O!GETi,4.500000,100.000000,0.556720



Reference Product: Argan Magic Color Protection Hair Mask – Nourishes, Hydrates & Helps to Protect Color Treated Hair | Enriched with Moringa Oil & Botanical Extracts | Made in USA | Paraben Free | Cruelty Free (8 oz)


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B00O2FGBJS,Argan Magic 100% Pure Argan Oil – Moisturizing...,Argan Magic,3.866667,60.000000,0.679818
1,B08WCG4JCC,"GIOVANNI Hemp Hydrating Hair Shine Spray, 4.3 ...",GIOVANNI,4.500000,90.000000,0.614432
2,B00OXDWFG2,"Nadira Organics Virgin Argan Oil for Skin, Fac...",UNGLINGA,4.818182,100.000000,0.614302
3,B012Q9NGE4,"Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths ...",Pure Body Naturals,4.625000,93.750000,0.602267
4,B09NSK1NQH,Dry Weightless Hair Volume and Texture Spray |...,Osensia,4.363636,81.818182,0.516145



Reference Product: AuroTrends Makeup Remover Towel,2 Pack Reusable Chemical-free Microfiber Makeup Remover Cloths,Magically and Gently Wipe Away Cosmetics,Sunscreen,Oil and Dirt,Set of 2 (Pink+Black)


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B088LWRYJC,NATWAG Reusable Update Makeup Remover Cloths f...,Natwag,4.823529,94.117647,0.638766
1,B0851T7HDS,MAD | Cosmetic's Makeup Remover Cleansing Towe...,JUST FOR TOTS BY MAD,3.090909,36.363636,0.620863
2,B0914JBBTV,Bioré J-Beauty Makeup Removing Moisturizing Cl...,Bioré,4.423077,88.461538,0.563084
3,B087F14JSH,"Natwag Makeup Remover Pads,18 Pack 3.74 Inch R...",Natwag,4.777778,100.000000,0.537333
4,B07KXM94BT,Makeup Blender Sponge Set - 4 Pack Latex Free ...,FRESHME,4.363636,81.818182,0.513654



Reference Product: Water Flosser Cordless Teeth Cleaner, SOOCAS Professional Oral Irrigator 230ML Portable and Rechargeable Water Dental Flosser with 3 Modes and 4 Jet Tips, IPX7 Waterproof, USB Charged for 30 Days Use


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B08BRS98SF,Wonder4 Electric Callus Remover - Rechargeable...,Wonder4,3.937500,75.000000,0.499476
1,B08HGZXLP6,Empty Amber Glass Spray Bottles - (4 Pack) 16 ...,AYL,4.869565,100.000000,0.438660
2,B09G22586Y,"Automatic Curling Iron, Rechargeable Cordless ...",LUZZAM,4.000000,70.000000,0.437212
3,B07PPLGRM2,JOYYUM 1000W 2-in-1 Hot Air Spin Brush Styler ...,JOYYUM,3.454545,45.454545,0.433899
4,B0851T7HDS,MAD | Cosmetic's Makeup Remover Cleansing Towe...,JUST FOR TOTS BY MAD,3.090909,36.363636,0.423960


# BERT: Sentence-BERT Evaluation Findings

## Description:

**Sentence-BERT demonstrated strong recommendation performance for several product categories, although its effectiveness varied depending on the reference product.**

- **Strengths**: Sentence-BERT identified relevant products across skincare, hair care, and makeup removal. It performed particularly well for the makeup remover towel, where four of the five recommendations were directly related to makeup removal. It also identified closely related argan oil, hair-care, and retinol products.

- **Limitations**: Recommendations for the water flosser were largely unrelated to oral care, indicating difficulty distinguishing product functionality in this example.

- **Similarity Interepretation**: Sentence-BERT produced a lower median cosine similarity than Word2Vec and GloVe, suggesting greater differentiation across product pairs. However, similarity scores alone do not establish recommendation relevance.

**Overall, Sentence-BERT demonstrated promising results, particularly when product descriptions contained meaningful contextual information about ingredients, intended uses, and product categories. Its performance was not consistent across every reference product, highlighting the importance of evaluating recommendations across multiple product types.**

# Compartive Analysis

## Description:

This section compares five product representation and similarity approaches: Original TF-IDF, Enhanced TF-IDF, Word2Vec, GloVe, and Sentence-BERT.

Each method is evaluated using the same five reference products to examine recommendation relevance, identify strengths and limitations, and assess how effectively each approach captures product similarity.

## Why start here:

It establishes the purpose of the final section and gives the reader context before we present the comparison

# Comparitive Analysis: Evaluation Criteria

##Description:

Recommendation quality is assessed qualitatively based on the relevance of the top five recommended products for each reference product.

The evaluation considers:
- Functional relevance: Whether recommended products serve a similar purpose.

- Category relevance: Whether recommended products belong to the same or a closely related product category.

- Ingredient and attribute relevance: Whether recommended products share meaningful ingredients or product characteristics.

The evaluation uses the same five reference products across all similarity methods to maintain consistency. This is a qualitative assessment rather than a formal precision or recall evaluation, since labeled relevance judgments are not available.



In [74]:
# Compare recommendation performance
comparison = pd.DataFrame(
    {
        "Argan Oil": ["Strong", "Strong", "Moderate"],
        "Retinol Moisturizer": ["Moderate", "Strong", "Strong"],
        "Hair Mask": ["Strong", "Strong", "Strong"],
        "Makeup Remover Towel": ["Strong", "Strong", "Strong"],
        "Water Flosser": ["Weak", "Weak", "Weak"]
    },
    index=[
        "Original TF-IDF",
        "Enhanced TF-IDF",
        "Word2Vec"
    ]
)

display(comparison)

,Argan Oil,Retinol Moisturizer,Hair Mask,Makeup Remover Towel,Water Flosser
Original TF-IDF,Strong,Moderate,Strong,Strong,Weak
Enhanced TF-IDF,Strong,Strong,Strong,Strong,Weak
Word2Vec,Moderate,Strong,Strong,Strong,Weak


In [70]:
#testing something
print("Original TF-IDF:", similarity_matrix.shape)
print("Enhanced TF-IDF:", similarity_enhanced_matrix.shape)

Original TF-IDF: (239, 239)
Enhanced TF-IDF: (239, 239)


In [71]:
#evaluate remaining 4 products: original tfidf
for product_id in evaluation_product_ids[1:]:
    print(f"\nReference Product: {product_id}")
    display(
        evaluate_recommendations(
            similarity_matrix,
            product_id,
            top_n=5
        )
    )


Reference Product: B00KCTER3U


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B08HVRP54L,So'Bio Étic | Moisturizing Fresh Gel Cream | O...,So'Bio Étic,4.100000,60.000000,0.189521
1,B083B67373,Avajar Blue LED Hydrating Cream Mask 5 Pack - ...,a DAILY SELF-CARE COSMETICS avajar,4.100000,90.000000,0.176951
2,B0B4JPGX8P,Elli K Essential Sincerity From AZ Time Revers...,ELLI K,4.076923,84.615385,0.158388
3,B08NJ5BTWG,FASCY LAB GREEN+ Korean Sunscreen — SPF 50+PA+...,Fascy,4.923077,100.000000,0.144834
4,B0851QJPZY,Gentlehomme Face Moisturizer For Men - Mens Fa...,Gentlehomme,4.100000,90.000000,0.133766



Reference Product: B07L9H27SH


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B00O2FGBJS,Argan Magic 100% Pure Argan Oil – Moisturizing...,Argan Magic,3.866667,60.000000,0.472415
1,B00OXDWFG2,"Nadira Organics Virgin Argan Oil for Skin, Fac...",UNGLINGA,4.818182,100.000000,0.176168
2,B07THLR7RR,LANGE HAIR Extendé Conditioning Detangler Hair...,NaN,4.166667,66.666667,0.165499
3,B08WCG4JCC,"GIOVANNI Hemp Hydrating Hair Shine Spray, 4.3 ...",GIOVANNI,4.500000,90.000000,0.163827
4,B09TMRB5NT,EMILIE HEATHE - Lip Atelier Lip Color | Sustai...,NaN,4.500000,91.666667,0.148068



Reference Product: B071R2QPF3


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B088LWRYJC,NATWAG Reusable Update Makeup Remover Cloths f...,Natwag,4.823529,94.117647,0.304308
1,B087F14JSH,"Natwag Makeup Remover Pads,18 Pack 3.74 Inch R...",Natwag,4.777778,100.000000,0.169610
2,B09TTC9Q9W,mold remover gel (3 Pack),Yondarli,3.000000,41.666667,0.147279
3,B0B1T39KQM,"9 Pcs Silicone Makeup Brush Cover, Reusable Du...",QrBxa,3.500000,40.000000,0.143946
4,B0BR4GL7Q6,Vegan Mia - USDA Organic Radiance Superfruit C...,Vegan Mia,4.136364,77.272727,0.140512



Reference Product: B07LCHCD6Q


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B086VRRD99,"C'est Moi Gentle Cleansing Water | Gentle, Fra...",C'est Moi,4.083333,75.000000,0.107966
1,B09G22586Y,"Automatic Curling Iron, Rechargeable Cordless ...",LUZZAM,4.000000,70.000000,0.089559
2,B087J3H22J,NEW Challenger Men's Rinse Free Hand Wash - 1o...,Challenger,4.333333,91.666667,0.079558
3,B08KGVBW41,Kneipp Shower Foam Goodbye Stress Water Mint &...,Kneipp,4.272727,72.727273,0.076234
4,B088LPYXNQ,20pcs Nail Tips Clips for Quick Building Polyg...,tiggell,4.250000,75.000000,0.058226


In [73]:
#evaluate remaining 4 products: enhanced tfidf
for product_id in evaluation_product_ids[1:]:
    print(f"\nReference Product: {product_id}")
    display(
        evaluate_recommendations(
            similarity_enhanced_matrix,
            product_id,
            top_n=5
        )
    )


Reference Product: B00KCTER3U


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B08HVRP54L,So'Bio Étic | Moisturizing Fresh Gel Cream | O...,So'Bio Étic,4.100000,60.000000,0.291235
1,B07W1WJZFG,"Simply Dana Cellular De-Stressing Cream, Boost...",Simply Dana,4.166667,75.000000,0.270442
2,B083B67373,Avajar Blue LED Hydrating Cream Mask 5 Pack - ...,a DAILY SELF-CARE COSMETICS avajar,4.100000,90.000000,0.255801
3,B08SQZ1B3W,Murad Hydration Intense Recovery Cream - Deepl...,NaN,4.500000,83.333333,0.242407
4,B0B5YVX9YY,CELLEANSER Vegan Clean Cream 100% Vegan Ingred...,Celleanser,4.909091,100.000000,0.234747



Reference Product: B07L9H27SH


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B0BXB4J297,Moroccanoil Color Care Conditioner,NaN,4.428571,85.714286,0.400257
1,B012Q9NGE4,"Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths ...",Pure Body Naturals,4.625000,93.750000,0.294255
2,B00O2FGBJS,Argan Magic 100% Pure Argan Oil – Moisturizing...,Argan Magic,3.866667,60.000000,0.284724
3,B07THLR7RR,LANGE HAIR Extendé Conditioning Detangler Hair...,NaN,4.166667,66.666667,0.265925
4,B00OXDWFG2,"Nadira Organics Virgin Argan Oil for Skin, Fac...",UNGLINGA,4.818182,100.000000,0.180511



Reference Product: B071R2QPF3


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B088LWRYJC,NATWAG Reusable Update Makeup Remover Cloths f...,Natwag,4.823529,94.117647,0.333326
1,B09TTC9Q9W,mold remover gel (3 Pack),Yondarli,3.000000,41.666667,0.196582
2,B087F14JSH,"Natwag Makeup Remover Pads,18 Pack 3.74 Inch R...",Natwag,4.777778,100.000000,0.195672
3,B08Q3F3L7S,Microfiber Hair Towels - 5 Towel Set – 5 Beaut...,AViBNiUM,4.700000,100.000000,0.161164
4,B0914JBBTV,Bioré J-Beauty Makeup Removing Moisturizing Cl...,Bioré,4.423077,88.461538,0.154444



Reference Product: B07LCHCD6Q


,item_id,title,store,avg_review_rating,positive_pct,similarity_score
0,B088LPYXNQ,20pcs Nail Tips Clips for Quick Building Polyg...,tiggell,4.250000,75.000000,0.105335
1,B086VRRD99,"C'est Moi Gentle Cleansing Water | Gentle, Fra...",C'est Moi,4.083333,75.000000,0.080196
2,B09G22586Y,"Automatic Curling Iron, Rechargeable Cordless ...",LUZZAM,4.000000,70.000000,0.079003
3,B08BLX72TP,LeiCare Tweezers Precision 2-Pack - Stainless ...,LeiCare,4.545455,90.909091,0.072283
4,B087J3H22J,NEW Challenger Men's Rinse Free Hand Wash - 1o...,Challenger,4.333333,91.666667,0.071618


In [75]:
#adding the GloVe and Sentence BERT to comparison table
comparison.loc["GloVe"] = [
    "Strong", "Moderate", "Moderate", "Weak", "Weak"
]

comparison.loc["Sentence-BERT"] = [
    "Strong", "Strong", "Strong", "Very strong", "Weak"
]

display(comparison)

,Argan Oil,Retinol Moisturizer,Hair Mask,Makeup Remover Towel,Water Flosser
Original TF-IDF,Strong,Moderate,Strong,Strong,Weak
Enhanced TF-IDF,Strong,Strong,Strong,Strong,Weak
Word2Vec,Moderate,Strong,Strong,Strong,Weak
GloVe,Strong,Moderate,Moderate,Weak,Weak
Sentence-BERT,Strong,Strong,Strong,Very strong,Weak


# Comparitive Analysis: Comparitive Analysis Findings

## Descriptions:

**The five product representation approaches demonstrated different strengths and limitations when identifying similar products across the evaluation set**

- Original TF-IDF: Identified relevant products through shared product terminology, particularly for argan oil, hair masks, and makeup removal. However, recommendations were less consistent for the retinol moisturizer and water flosser.

- Enhanced TF-IDF: Demonstrated consistent performance across four reference products. Incorporating additional product attributes helped identify relevant skincare, hair-care, and makeup-removal products. However, it continued to struggle with the water flosser.

- Word2Vec: Captured broader semantic relationships between products, particularly across skincare and hair-care categories. However, some recommendations were unrelated to the reference product's intended function.

- GloVe: Identified broad semantic associations involving ingredients and beauty-related terminology. However, its recommendations were less consistent in distinguishing products with similar vocabulary from products with similar functions.

- Sentence-BERT: Demonstrated strong results across skincare, hair care, and makeup removal. It performed particularly well for the makeup remover towel, where all five recommendations were related to makeup removal or application. However, it struggled with the water flosser.

**Overall Observations**:

Enhanced TF-IDF and Sentence-BERT demonstrated the most consistent qualitative relevance across the five reference products. Enhanced TF-IDF benefited from incorporating additional product attributes, while Sentence-BERT captured contextual relationships beyond exact keyword overlap.

The water flosser was a consistent limitation across all five approaches, suggesting that product functionality and category-specific terminology may require additional attention in the recommendation process.

These findings are based on qualitative inspection of the top five recommendations rather than formal precision or recall metrics. Since labeled relevance judgments are unavailable, the results should be interpreted as exploratory comparisons rather than definitive performance rankings.

# CONCLUSION: Transitioning to Recommendation Modeling

This notebook explored several approaches for representing products and measuring similarity using product titles, descriptions, and additional attributes.

The comparative analysis highlighted the strengths and limitations of traditional text-based methods and pretrained embedding approaches. Enhanced TF-IDF and Sentence-BERT demonstrated promising qualitative results, although recommendation relevance varied across product categories.

The findings also highlighted the limitations of relying on product text alone. Similarity between product descriptions does not necessarily guarantee functional relevance or indicate product quality.

## NEXT STEPS: Hybrid Recommendation Model

**The next notebook will build upon these findings by developing a hybrid product recommendation model that combines:**

- **Product Similarity**: Identifies products with similar characteristics and descriptions.

- **Customer Ratings**: Incorporates average product ratings to account for customer satisfaction.

- **Review Sentiment**: Uses positive customer feedback to help identify well-received products.

By combining these signals, the recommendation model will aim to provide more relevant and useful product recommendations than relying on textual similarity alone.